# PokerBench GTO Fine-Tuning — Local (VSCode) Version

**Project:** CIS 4270/5270 Trustworthy ML — Spring 2026
**Team:** Max Mercado, Andrew Wang
**Task:** GTO action prediction in 6-handed No-Limit Texas Hold'em

This notebook fine-tunes GPT models on PokerBench with two algorithms:

1. **SFT** — Supervised Fine-Tuning (imitate solver-optimal actions)
2. **DPO** — Direct Preference Optimization (learn from (correct, incorrect) pairs)

**RFT is on hold** (course-wide budget constraint). The RFT code is preserved
but gated behind a flag that's off by default.

> **How to run:**
> - First time: `pip install -r requirements.txt`
> - Top-to-bottom works; the flags in Section 0 control what actually runs.
> - State persists across kernel restarts via `finetuned_state.json`.
> - Large data files cache to `./cache/` so subsequent reloads are fast.

## 0. Configuration

**The only cell you edit between runs.**

In [ ]:
# =============================================================================
# FEATURE FLAGS
# =============================================================================

# ----- Run version (v2 — Class B + sizing tolerance changes) -----
RUN_VERSION = "v2"

# ----- Data prep -----
REBUILD_DATA   = True     # need new DPO data with Class B negatives
REUPLOAD_SFT   = True     # fresh file IDs in the new state file
REUPLOAD_DPO   = True
REUPLOAD_RFT   = False

# ----- Training -----
TRAIN_SFT      = False    # sweep mode handles both
TRAIN_DPO      = False
RUN_RFT        = False

# ----- Hyperparameter sweep -----
DO_SWEEP       = True     # both SFT and DPO go through the grid

# ----- Deployment -----
# Run deploy + eval + delete inside one 4h window.
DEPLOY_SFT     = True
DEPLOY_DPO     = True
DEPLOY_RFT     = False

# ----- Evaluation -----
RUN_BASELINE   = True
RUN_FULL_EVAL  = False

# ----- Bet sizing tolerance -----
BET_SIZE_TOL   = 0.10     # within 10% of correct sizing counts as correct

# =============================================================================
# DATA SIZES
# =============================================================================
N_SFT_TRAIN = 2000
N_SFT_VAL   = 400
N_DPO_TRAIN = 2000
N_DPO_VAL   = 400
N_RFT_TRAIN = 1000
N_RFT_VAL   = 200
N_TEST      = 500
N_BASELINE  = 100

PREFLOP_FRACTION = 0.5
POSTFLOP_FILTER  = "balanced"
RANDOM_SEED      = 42

# =============================================================================
# PATHS
# =============================================================================
import os
CACHE_DIR = "./cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SFT_TRAIN_PATH = f"sft_train_{RUN_VERSION}.jsonl"
SFT_VAL_PATH   = f"sft_val_{RUN_VERSION}.jsonl"
DPO_TRAIN_PATH = f"dpo_train_{RUN_VERSION}.jsonl"
DPO_VAL_PATH   = f"dpo_val_{RUN_VERSION}.jsonl"
RFT_TRAIN_PATH = f"rft_train_{RUN_VERSION}.jsonl"
RFT_VAL_PATH   = f"rft_val_{RUN_VERSION}.jsonl"
TEST_PATH      = f"test_set_{RUN_VERSION}.jsonl"

PERSIST_PATH = f"finetuned_state_{RUN_VERSION}.json"

# =============================================================================
# PROMPT VERSION (current run)
# =============================================================================
PROMPT_VERSION = "v2_no_system"
SYSTEM_PROMPT  = ""

BASELINE_OVERRIDE = None

print(f"Flags: REBUILD_DATA={REBUILD_DATA}  TRAIN_SFT={TRAIN_SFT}  TRAIN_DPO={TRAIN_DPO}  DO_SWEEP={DO_SWEEP}  RUN_RFT={RUN_RFT}")
print(f"       DEPLOY: SFT={DEPLOY_SFT}  DPO={DEPLOY_DPO}  RFT={DEPLOY_RFT}")
print(f"Prompt: {PROMPT_VERSION}")

## 1. Setup Check

Run this once per machine:

```bash
pip install -r requirements.txt
az login
```

## 1.1 Azure Client

In [ ]:
import os, json, random, re, pickle
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import (
    Deployment, DeploymentProperties, DeploymentModel, Sku,
)

random.seed(RANDOM_SEED)

In [ ]:
RESOURCE_GROUP  = os.environ["AZURE_RESOURCE_GROUP"]   # set this env var before running
OPENAI_API_KEY  = os.environ["AZURE_OPENAI_API_KEY"]  # set this env var before running
OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]  # set this env var before running

SFT_DPO_BASE_MODEL = "<your-sft-dpo-base-model>"
RFT_BASE_MODEL     = "<your-rft-base-model>"

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = RESOURCE_GROUP
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT
CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)
print(f"Connected to {OPENAI_ENDPOINT}")

### 2 State and prompt helpers

Fine-tuned models are stored with the prompt regime they were trained under,
so evaluation reconstructs the right message format per-model.

In [ ]:
def load_state(path=PERSIST_PATH):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return {}

def save_state(updates, path=PERSIST_PATH):
    state = load_state(path)
    state.update(updates)
    with open(path, "w") as f:
        json.dump(state, f, indent=2)
    print(f"state <- {list(updates.keys())}")
    return state

# One-time migration: legacy state used 'dpo' instead of 'dpo_model_id'
_raw_state = load_state()
if "dpo" in _raw_state and "dpo_model_id" not in _raw_state:
    print("Migrating legacy 'dpo' -> 'dpo_model_id'...")
    save_state({
        "dpo_model_id":       _raw_state["dpo"],
        "dpo_prompt_version": _raw_state.get("dpo_prompt_version", "v1_long_system"),
    })

PROMPT_REGIMES = {
    "v1_long_system": (
        "You are an expert poker player specializing in game-theoretically optimal."
        "Given a game scenario, respond with only the optimal action \u2014 no explanation, no punctuation. "
        "Valid formats: fold, call, check, Bet <amount>, Raise <amount>, or <amount>bb for preflop raises."
        ),
    "v2_long_system": (
        "You are an expert poker player specializing in game-theoretically optimal (GTO) "
        "play for 6-handed No-Limit Texas Hold'em. Given a game scenario, output the single "
        "best action exactly as it should appear — no explanation, no punctuation. "
        "Examples of valid outputs: 'fold', 'call', 'check', 'Bet 24', 'Raise 48'."
    ),
    "v3_no_system": "",
}

def build_messages(prompt, prompt_version=None, flatten=False):
    """Assemble chat messages for inference.
    - prompt_version defaults to the current run's PROMPT_VERSION when not passed.
    - flatten=True folds the system prompt into the user message (RFT format)."""
    if prompt_version is None:
        prompt_version = PROMPT_VERSION
    sys_text = PROMPT_REGIMES.get(prompt_version, "")
    if not sys_text:
        return [{"role": "user", "content": prompt}]
    if flatten:
        return [{"role": "user", "content": f"{sys_text}\n\n{prompt}"}]
    return [
        {"role": "system", "content": sys_text},
        {"role": "user",   "content": prompt},
    ]

print("State + prompt helpers ready.")

In [ ]:
# Load saved state into working variables
_state = load_state()

sft_model_id       = _state.get("sft_model_id")
dpo_model_id       = _state.get("dpo_model_id")
rft_model_id       = _state.get("rft_model_id")
sft_prompt_version = _state.get("sft_prompt_version", PROMPT_VERSION)
dpo_prompt_version = _state.get("dpo_prompt_version", PROMPT_VERSION)
rft_prompt_version = _state.get("rft_prompt_version", PROMPT_VERSION)
sft_deployment     = _state.get("sft_deployment")
dpo_deployment     = _state.get("dpo_deployment")
rft_deployment     = _state.get("rft_deployment")
baseline_acc       = _state.get("baseline_acc")
sft_job_id         = _state.get("sft_job_id")
dpo_job_id         = _state.get("dpo_job_id")
rft_job_id         = _state.get("rft_job_id")

if BASELINE_OVERRIDE is not None:
    baseline_acc = BASELINE_OVERRIDE

if _state:
    print("Loaded state:")
    print(f"  SFT: model={sft_model_id}")
    print(f"       prompt={sft_prompt_version}  deploy={sft_deployment}")
    print(f"  DPO: model={dpo_model_id}")
    print(f"       prompt={dpo_prompt_version}  deploy={dpo_deployment}")
    print(f"  RFT: model={rft_model_id}")
    print(f"       prompt={rft_prompt_version}  deploy={rft_deployment}")
    print(f"  Baseline: {baseline_acc}")
else:
    print(f"No state at {PERSIST_PATH} — fresh run.")

## 3. Load PokerBench

First run downloads ~1.6 GB from HuggingFace. Subsequent runs load from
`./cache/pokerbench_raw.pkl`.

In [ ]:
from datasets import load_dataset
import pandas as pd

HF_REPO = "RZ412/PokerBench"
RAW_CACHE = os.path.join(CACHE_DIR, "pokerbench_raw.pkl")

def _download_pokerbench():
    print("Downloading PokerBench from HuggingFace (one-time, ~1.6 GB)...")
    dfs = {
        "preflop_train_df": pd.read_json(
            f"hf://datasets/{HF_REPO}/preflop_60k_train_set_prompt_and_label.json"),
        "preflop_train_csv": pd.read_csv(
            f"hf://datasets/{HF_REPO}/preflop_60k_train_set_game_scenario_information.csv"),
        "postflop_train_df": pd.read_json(
            f"hf://datasets/{HF_REPO}/postflop_500k_train_set_prompt_and_label.json"),
        "postflop_train_csv": pd.read_csv(
            f"hf://datasets/{HF_REPO}/postflop_500k_train_set_game_scenario_information.csv"),
        "preflop_test_df": pd.read_json(
            f"hf://datasets/{HF_REPO}/preflop_1k_test_set_prompt_and_label.json"),
        "preflop_test_csv": pd.read_csv(
            f"hf://datasets/{HF_REPO}/preflop_1k_test_set_game_scenario_information.csv"),
        "postflop_test_df": pd.read_json(
            f"hf://datasets/{HF_REPO}/postflop_10k_test_set_prompt_and_label.json"),
        "postflop_test_csv": pd.read_csv(
            f"hf://datasets/{HF_REPO}/postflop_10k_test_set_game_scenario_information.csv"),
    }
    with open(RAW_CACHE, "wb") as f:
        pickle.dump(dfs, f)
    print(f"Cached raw frames -> {RAW_CACHE}")
    return dfs

if REBUILD_DATA or not os.path.exists(RAW_CACHE):
    _raw = _download_pokerbench()
else:
    print(f"Loading PokerBench from cache: {RAW_CACHE}")
    with open(RAW_CACHE, "rb") as f:
        _raw = pickle.load(f)

preflop_train_df   = _raw["preflop_train_df"]
preflop_train_csv  = _raw["preflop_train_csv"]
postflop_train_df  = _raw["postflop_train_df"]
postflop_train_csv = _raw["postflop_train_csv"]
preflop_test_df    = _raw["preflop_test_df"]
preflop_test_csv   = _raw["preflop_test_csv"]
postflop_test_df   = _raw["postflop_test_df"]
postflop_test_csv  = _raw["postflop_test_csv"]

print(f"Preflop train:  {len(preflop_train_df):>7,} prompts, {len(preflop_train_csv):>7,} CSV rows")
print(f"Postflop train: {len(postflop_train_df):>7,} prompts, {len(postflop_train_csv):>7,} CSV rows")
print(f"Preflop test:   {len(preflop_test_df):>7,}")
print(f"Postflop test:  {len(postflop_test_df):>7,}")

### 3.1 Inspect schema

Sanity check. Skip if you've run before with the same dataset version.

In [ ]:
print("=== Preflop JSON (first row) ===")
print(json.dumps(preflop_train_df.iloc[0].to_dict(), indent=2)[:500])
print()
print("=== Preflop CSV (first row) ===")
print(preflop_train_csv.iloc[0].to_dict())
print()
print("=== Postflop CSV (first row) ===")
print(postflop_train_csv.iloc[0].to_dict())

## 4. Data Preparation

Transforms PokerBench rows into a unified format, then writes three JSONLs.
Skipped entirely when `REBUILD_DATA=False`.

### 4.1 Action parser

In [ ]:
def parse_action(action_str):
    """Parse a poker action string into (category, amount).
    category in {fold, call, check, allin, bet, raise, unknown}
    amount is float or None.
    """
    s = str(action_str).strip().lower()
    if s == "fold":  return ("fold",  None)
    if s == "call":  return ("call",  None)
    if s == "check": return ("check", None)
    if s in ("allin", "all-in", "all in"): return ("allin", None)
    m = re.match(r"^(bet|raise)\s+([0-9.]+)", s)
    if m:
        return (m.group(1), float(m.group(2)))
    m = re.match(r"^([0-9.]+)\s*bb$", s)
    if m:
        return ("raise", float(m.group(1)))
    return ("unknown", None)

def action_tuple(s):
    """parse_action variant that strips trailing punctuation for eval comparison."""
    s = str(s).strip().rstrip(".,!? \n").lower()
    return parse_action(s)

def actions_match(pred_str, correct_str, tol=None):
    """True if categories match AND amounts are within `tol` (relative)."""
    if tol is None:
        tol = BET_SIZE_TOL
    p_cat, p_amt = action_tuple(pred_str)
    c_cat, c_amt = action_tuple(correct_str)
    if p_cat != c_cat or p_cat == "unknown":
        return False
    if p_amt is None and c_amt is None:
        return True
    if p_amt is None or c_amt is None:
        return False
    return abs(p_amt - c_amt) / max(c_amt, 1.0) <= tol

for a in ["fold", "Call", "Bet 24", "raise 14", "2.0bb", "allin"]:
    print(f"{a:>10}  ->  {parse_action(a)}")


### 4.2 Build unified rows

In [ ]:
import ast

def parse_available_moves(raw):
    if isinstance(raw, list):
        return raw
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []

def build_unified_rows(prompt_df, csv_df, stage_tag):
    assert len(prompt_df) == len(csv_df), \
        f"Row count mismatch: JSON={len(prompt_df)} vs CSV={len(csv_df)}"
    rows = []
    for i in range(len(prompt_df)):
        pj = prompt_df.iloc[i]
        pc = csv_df.iloc[i]
        available = parse_available_moves(pc["available_moves"])
        if len(available) < 2:
            continue
        if stage_tag == "preflop":
            street = "preflop"
        else:
            raw_street = str(pc.get("evaluation_at", "")).strip().lower()
            street = raw_street if raw_street in ("flop", "turn", "river") else "flop"
        rows.append({
            "prompt":           pj["instruction"],
            "correct_decision": pj["output"],
            "available_moves":  available,
            "stage":            stage_tag,
            "street":           street,
        })
    return rows

UNIFIED_CACHE = os.path.join(CACHE_DIR, "pokerbench_unified.pkl")

if REBUILD_DATA or not os.path.exists(UNIFIED_CACHE):
    print("Building unified rows...")
    _unified = {
        "preflop_train_rows":  build_unified_rows(preflop_train_df,  preflop_train_csv,  "preflop"),
        "postflop_train_rows": build_unified_rows(postflop_train_df, postflop_train_csv, "postflop"),
        "preflop_test_rows":   build_unified_rows(preflop_test_df,   preflop_test_csv,   "preflop"),
        "postflop_test_rows":  build_unified_rows(postflop_test_df,  postflop_test_csv,  "postflop"),
    }
    with open(UNIFIED_CACHE, "wb") as f:
        pickle.dump(_unified, f)
    print(f"Cached unified rows -> {UNIFIED_CACHE}")
else:
    print(f"Loading unified rows from cache: {UNIFIED_CACHE}")
    with open(UNIFIED_CACHE, "rb") as f:
        _unified = pickle.load(f)

preflop_train_rows  = _unified["preflop_train_rows"]
postflop_train_rows = _unified["postflop_train_rows"]
preflop_test_rows   = _unified["preflop_test_rows"]
postflop_test_rows  = _unified["postflop_test_rows"]

print(f"Preflop train usable:  {len(preflop_train_rows):>7,}")
print(f"Postflop train usable: {len(postflop_train_rows):>7,}")
print(f"Test usable:           {len(preflop_test_rows) + len(postflop_test_rows):>7,}")

### 4.3 Filter + sample

In [ ]:
def filter_postflop(rows, mode):
    if mode == "flop_only":
        return [r for r in rows if r["street"] == "flop"]
    if mode == "balanced":
        by_street = {"flop": [], "turn": [], "river": []}
        for r in rows:
            s = r["street"]
            if s in by_street:
                by_street[s].append(r)
        min_n = min(len(v) for v in by_street.values())
        out = []
        for v in by_street.values():
            random.shuffle(v)
            out.extend(v[:min_n])
        return out
    return rows

random.seed(RANDOM_SEED)
postflop_train_rows = filter_postflop(postflop_train_rows, POSTFLOP_FILTER)
print(f"After postflop filter ({POSTFLOP_FILTER}): {len(postflop_train_rows):,} rows")

random.shuffle(preflop_train_rows)
random.shuffle(postflop_train_rows)

def mix_and_take(n, pre, post, frac):
    n_pre  = int(n * frac)
    n_post = n - n_pre
    sample = pre[:n_pre] + post[:n_post]
    random.shuffle(sample)
    return sample

total_needed = (N_SFT_TRAIN + N_SFT_VAL +
                N_DPO_TRAIN + N_DPO_VAL +
                N_RFT_TRAIN + N_RFT_VAL)
pool = mix_and_take(total_needed, preflop_train_rows, postflop_train_rows, PREFLOP_FRACTION)
print(f"Combined training pool: {len(pool):,} rows")

def take_split(pool_rows, n_train, n_val, offset=0):
    total = n_train + n_val
    chunk = pool_rows[offset:offset + total]
    return chunk[:n_train], chunk[n_train:]

sft_train, sft_val = take_split(pool, N_SFT_TRAIN, N_SFT_VAL, offset=0)
dpo_train, dpo_val = take_split(pool, N_DPO_TRAIN, N_DPO_VAL, offset=0)
rft_train, rft_val = take_split(pool, N_RFT_TRAIN, N_RFT_VAL, offset=0)

test_pool = preflop_test_rows + postflop_test_rows
random.shuffle(test_pool)
test_rows = test_pool[:N_TEST]

for name, split in [("SFT train", sft_train), ("SFT val", sft_val),
                    ("DPO train", dpo_train), ("DPO val", dpo_val),
                    ("RFT train", rft_train), ("RFT val", rft_val),
                    ("Test",      test_rows)]:
    print(f"{name:>10}: {len(split):,}")

### 4.4 Write SFT JSONL

In [ ]:
def write_sft_jsonl(rows, path):
    with open(path, "w") as f:
        for r in rows:
            messages = []
            if SYSTEM_PROMPT:
                messages.append({"role": "system", "content": SYSTEM_PROMPT})
            messages.append({"role": "user",      "content": r["prompt"]})
            messages.append({"role": "assistant", "content": str(r["correct_decision"])})
            f.write(json.dumps({"messages": messages}) + "\n")

if REBUILD_DATA:
    write_sft_jsonl(sft_train, SFT_TRAIN_PATH)
    write_sft_jsonl(sft_val,   SFT_VAL_PATH)
    print(f"Wrote {SFT_TRAIN_PATH} ({len(sft_train)}) and {SFT_VAL_PATH} ({len(sft_val)})")
    with open(SFT_TRAIN_PATH) as f:
        print("--- first record ---")
        print(json.dumps(json.loads(f.readline()), indent=2)[:500])
else:
    print("REBUILD_DATA=False: skipping SFT JSONL write.")

### 4.5 Write DPO JSONL

Negative selection per proposal: prefer Class C (wrong category) over Class B
(wrong sizing). Within Class C, tie-break by heuristic strategic-loss score.

In [ ]:
def _strategic_loss(correct_cat, correct_amt, action):
    a_cat, a_amt = parse_action(action)
    # Class C (wrong category) — worst offenders get higher weight
    if correct_cat in ("bet", "raise") and a_cat == "fold": return 4
    if correct_cat in ("bet", "raise") and a_cat in ("check", "call"): return 2
    if correct_cat in ("check", "call") and a_cat in ("bet", "raise"): return 3
    if correct_cat == "fold" and a_cat in ("bet", "raise"): return 4
    # Class B (right category, wrong size) — weight by sizing error,
    # but treat anything within tolerance as correct (weight 0)
    if a_cat == correct_cat and a_amt is not None and correct_amt is not None:
        rel = abs(a_amt - correct_amt) / max(correct_amt, 1.0)
        if rel <= BET_SIZE_TOL:  return 0    # within 10% → not a negative
        if rel <= 0.30:          return 1    # mildly off
        return 2                              # badly off
    return 1  # other Class C

def dpo_negatives_with_weights(correct, available):
    """Yield (negative_action, weight) for every legal incorrect action.
    Weight 0 = skip (counts as correct under tolerance)."""
    correct_cat, correct_amt = parse_action(correct)
    for a in available:
        if str(a).strip().lower() == str(correct).strip().lower():
            continue
        w = _strategic_loss(correct_cat, correct_amt, a)
        if w > 0:
            yield a, w

def write_dpo_jsonl(rows, path):
    written = 0
    with open(path, "w") as f:
        for r in rows:
            for neg, weight in dpo_negatives_with_weights(
                r["correct_decision"], r["available_moves"]
            ):
                input_messages = []
                if SYSTEM_PROMPT:
                    input_messages.append({"role": "system", "content": SYSTEM_PROMPT})
                input_messages.append({"role": "user", "content": r["prompt"]})
                record = {
                    "input": {"messages": input_messages},
                    "preferred_output":     [{"role": "assistant", "content": str(r["correct_decision"])}],
                    "non_preferred_output": [{"role": "assistant", "content": str(neg)}],
                }
                # Emphasis-on-worse: emit `weight` copies (3 for fold-vs-bet, 1 for mild B, etc.)
                for _ in range(weight):
                    f.write(json.dumps(record) + "\n")
                    written += 1
    return written

if REBUILD_DATA:
    dpo_train_n = write_dpo_jsonl(dpo_train, DPO_TRAIN_PATH)
    dpo_val_n   = write_dpo_jsonl(dpo_val,   DPO_VAL_PATH)
    print(f"Wrote {DPO_TRAIN_PATH} ({dpo_train_n}) and {DPO_VAL_PATH} ({dpo_val_n})")
    with open(DPO_TRAIN_PATH) as f:
        print("--- first record ---")
        print(json.dumps(json.loads(f.readline()), indent=2)[:800])
else:
    print("REBUILD_DATA=False: skipping DPO JSONL write.")

### 4.6 Write RFT JSONL (on hold)

In [ ]:
def write_rft_jsonl(rows, path):
    """Azure RFT rejects system messages; fold into user content when non-empty."""
    with open(path, "w") as f:
        for r in rows:
            user_content = f"{SYSTEM_PROMPT}\n\n{r['prompt']}" if SYSTEM_PROMPT else r["prompt"]
            record = {
                "messages":        [{"role": "user", "content": user_content}],
                "answer":          str(r["correct_decision"]),
                "available_moves": [str(a) for a in r["available_moves"]],
            }
            f.write(json.dumps(record) + "\n")

if REBUILD_DATA and RUN_RFT:
    write_rft_jsonl(rft_train, RFT_TRAIN_PATH)
    write_rft_jsonl(rft_val,   RFT_VAL_PATH)
    write_rft_jsonl(test_rows,       TEST_PATH)
    print(f"Wrote {RFT_TRAIN_PATH} / {RFT_VAL_PATH} / {TEST_PATH}")
else:
    print("Skipping RFT JSONL (RUN_RFT or REBUILD_DATA is False).")

## 5. Upload Training Files to Azure

Only runs when `REBUILD_DATA=True` and the per-algo REUPLOAD flag is on.
File IDs are persisted to state.

In [ ]:
def upload_file(path):
    with open(path, "rb") as f:
        up = openai_client.files.create(file=f, purpose="fine-tune")
    print(f"  {path} -> {up.id}")
    return up.id

sft_train_id = _state.get("sft_train_id")
sft_val_id   = _state.get("sft_val_id")
dpo_train_id = _state.get("dpo_train_id")
dpo_val_id   = _state.get("dpo_val_id")
rft_train_id = _state.get("rft_train_id")
rft_val_id   = _state.get("rft_val_id")

uploaded_ids = []

if REBUILD_DATA and REUPLOAD_SFT:
    print("Uploading SFT files...")
    sft_train_id = upload_file(SFT_TRAIN_PATH)
    sft_val_id   = upload_file(SFT_VAL_PATH)
    uploaded_ids += [sft_train_id, sft_val_id]

if REBUILD_DATA and REUPLOAD_DPO:
    print("Uploading DPO files...")
    dpo_train_id = upload_file(DPO_TRAIN_PATH)
    dpo_val_id   = upload_file(DPO_VAL_PATH)
    uploaded_ids += [dpo_train_id, dpo_val_id]

if REBUILD_DATA and REUPLOAD_RFT and RUN_RFT:
    print("Uploading RFT files...")
    rft_train_id = upload_file(RFT_TRAIN_PATH)
    rft_val_id   = upload_file(RFT_VAL_PATH)
    uploaded_ids += [rft_train_id, rft_val_id]

if uploaded_ids:
    print("Waiting for Azure file processing...")
    for fid in uploaded_ids:
        openai_client.files.wait_for_processing(fid)
    print("Files ready.")
    save_state({
        "sft_train_id": sft_train_id, "sft_val_id": sft_val_id,
        "dpo_train_id": dpo_train_id, "dpo_val_id": dpo_val_id,
        "rft_train_id": rft_train_id, "rft_val_id": rft_val_id,
    })
else:
    print("No files to upload.")
    print(f"Cached: sft_train={sft_train_id}  dpo_train={dpo_train_id}")

## 6. Baseline: un-fine-tuned `gpt-4.1-nano`

Only runs if `RUN_BASELINE=True` and no baseline is cached. Uses
`N_BASELINE=100` (sanity check, not full test).

In [ ]:
def evaluate_accuracy(model_deployment, rows, prompt_version=None,
                      flatten=False, n=None, verbose=False):
    """Exact-match accuracy on (category, amount) tuple."""
    n = n or len(rows)
    correct = 0
    errors  = 0
    for i, r in enumerate(rows[:n]):
        try:
            messages = build_messages(r["prompt"], prompt_version=prompt_version, flatten=flatten)
            resp = openai_client.chat.completions.create(
                model=model_deployment, messages=messages,
                max_tokens=20, temperature=0.0,
            )
            pred = resp.choices[0].message.content
            if action_tuple(pred) == action_tuple(r["correct_decision"]):
                correct += 1
            if verbose and i < 3:
                print(f"  [{i}] pred={pred!r}  correct={r['correct_decision']!r}")
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"  error row {i}: {e}")
    scored = n - errors
    acc = correct / max(scored, 1)
    print(f"Accuracy: {correct}/{scored} = {acc:.3f}  ({errors} errors)")
    return acc

if RUN_BASELINE:
    BASE_DEPLOY = "<your-sft-dpo-base-model>"  # change if your base deployment is named differently
    print(f"Evaluating base {BASE_DEPLOY} on {N_BASELINE} test rows...")
    try:
        baseline_acc = evaluate_accuracy(
            BASE_DEPLOY, test_rows,
            prompt_version=PROMPT_VERSION, n=N_BASELINE, verbose=True,
        )
        save_state({"baseline_acc": baseline_acc})
    except Exception as e:
        print(f"Baseline eval failed: {e}")
        print("Either deploy the base model or set BASELINE_OVERRIDE in Section 0.")
# elif baseline_acc is not None:
#     print(f"Baseline already cached: {baseline_acc:.3f}")
else:
    print("RUN_BASELINE=False, skipping.")

## 7. Fine-Tuning Jobs

Each algorithm is gated independently. By default we skip training and reuse
saved model IDs.

### 7.1 SFT

In [ ]:
SFT_HYPERPARAMETERS = {
    "n_epochs": 1,
    "batch_size": 4,
    "learning_rate_multiplier": 0.3,
}

if TRAIN_SFT and not DO_SWEEP:
    assert sft_train_id and sft_val_id, "SFT file IDs missing; set REBUILD_DATA + REUPLOAD_SFT."
    print(f"Creating SFT job on {SFT_DPO_BASE_MODEL}...")
    sft_job = openai_client.fine_tuning.jobs.create(
        model=SFT_DPO_BASE_MODEL,
        training_file=sft_train_id,
        validation_file=sft_val_id,
        method={
            "type": "supervised",
            "supervised": {"hyperparameters": SFT_HYPERPARAMETERS},
        },
        extra_body={"trainingType": "GlobalStandard"},
        suffix="poker-sft-v3",
    )
    print(f"SFT Job ID: {sft_job.id}  status: {sft_job.status}")
    sft_job_id = sft_job.id
    save_state({"sft_job_id": sft_job_id, "sft_prompt_version": PROMPT_VERSION})
else:
    print(f"TRAIN_SFT=False (or sweep mode). Using saved job: {sft_job_id}")

### 7.2 DPO

In [ ]:
DPO_HYPERPARAMETERS = {
    "n_epochs": 2,
    "batch_size": 4,
    "learning_rate_multiplier": 0.5,
}

if TRAIN_DPO and not DO_SWEEP:
    assert dpo_train_id and dpo_val_id, "DPO file IDs missing; set REBUILD_DATA + REUPLOAD_DPO."
    print(f"Creating DPO job on {SFT_DPO_BASE_MODEL}...")
    dpo_job = openai_client.fine_tuning.jobs.create(
        model=SFT_DPO_BASE_MODEL,
        training_file=dpo_train_id,
        validation_file=dpo_val_id,
        method={"type": "dpo", "dpo": {"hyperparameters": DPO_HYPERPARAMETERS}},
        extra_body={"trainingType": "GlobalStandard"},
        suffix="poker-dpo",
    )
    print(f"DPO Job ID: {dpo_job.id}  status: {dpo_job.status}")
    dpo_job_id = dpo_job.id
    save_state({"dpo_job_id": dpo_job_id, "dpo_prompt_version": PROMPT_VERSION})
else:
    print(f"TRAIN_DPO=False (or sweep mode). Using saved job: {dpo_job_id}")

### 7.3 RFT (ON HOLD)

Course-wide decision. Only runs if `RUN_RFT=True`.

In [ ]:
GRADER_SOURCE = '''
import re

def parse_action(action_str):
    s = str(action_str).strip().lower()
    if s == "fold":  return ("fold",  None)
    if s == "call":  return ("call",  None)
    if s == "check": return ("check", None)
    if s in ("allin", "all-in", "all in"): return ("allin", None)
    m = re.match(r"^(bet|raise)\\s+([0-9.]+)", s)
    if m: return (m.group(1), float(m.group(2)))
    m = re.match(r"^([0-9.]+)\\s*bb$", s)
    if m: return ("raise", float(m.group(1)))
    return ("unknown", None)

def grade(sample, item) -> float:
    raw = sample["output_text"] if "output_text" in sample else sample.get("output", "")
    model_out = str(raw).strip().rstrip(".,!? \\n")
    correct   = str(item["answer"]).strip()
    available = item.get("available_moves", []) or []

    m_cat, m_amt = parse_action(model_out)
    c_cat, c_amt = parse_action(correct)

    if available:
        legal_cats = {parse_action(a)[0] for a in available}
        if m_cat == "unknown" or m_cat not in legal_cats:
            return 0.0

    if m_cat == c_cat:
        if m_amt is None and c_amt is None:
            return 1.0
        if m_amt is not None and c_amt is not None:
            rel = abs(m_amt - c_amt) / max(c_amt, 1.0)
            if rel <= 0.10:   return 1.0   # within tolerance → full credit
            return 0.5                     # right category, wrong size
    return 0.0
'''

grader = {"type": "python", "name": "poker_quality_class", "source": GRADER_SOURCE}

RFT_HYPERPARAMETERS = {
    "n_epochs": 2,
    "batch_size": 6,
    "learning_rate_multiplier": 0.7,
    "eval_interval": 5,
    "eval_samples": 2,
    "reasoning_effort": "low",
}

if RUN_RFT and not DO_SWEEP:
    assert rft_train_id and rft_val_id, "RFT file IDs missing."
    print(f"Creating RFT job on {RFT_BASE_MODEL}...")
    rft_job = openai_client.fine_tuning.jobs.create(
        model=RFT_BASE_MODEL,
        training_file=rft_train_id,
        validation_file=rft_val_id,
        method={
            "type": "reinforcement",
            "reinforcement": {"grader": grader, "hyperparameters": RFT_HYPERPARAMETERS},
        },
        extra_body={"trainingType": "Standard"},
        suffix="poker-rft-v2",
    )
    print(f"RFT Job ID: {rft_job.id}  status: {rft_job.status}")
    rft_job_id = rft_job.id
    save_state({"rft_job_id": rft_job_id, "rft_prompt_version": PROMPT_VERSION})
else:
    print(f"RUN_RFT=False (course-wide hold). Saved job: {rft_job_id}")

### 7.4 Hyperparameter sweep (opt-in)

Per the course update: HP tuning is optional; if done, use `validation_file`
during training and deploy only the best-val model. This cell launches a grid
with validation attached. Selection happens in Section 8.2 and only the winner
gets deployed.

**Default grid: 2×2 per algorithm (4 SFT jobs + 4 DPO jobs).** Extend the
lists below for a 3×3 if budget allows.

In [ ]:
SWEEP_SFT_GRID = {
    "n_epochs":                 [1, 2],
    "learning_rate_multiplier": [0.3, 0.5],
    "batch_size":               [4],
}

SWEEP_DPO_GRID = {
    "n_epochs":                 [1, 2],
    "learning_rate_multiplier": [0.3, 0.5],
    "batch_size":               [4],
}

def _grid_product(grid):
    import itertools
    keys = list(grid.keys())
    for combo in itertools.product(*(grid[k] for k in keys)):
        yield dict(zip(keys, combo))

def launch_sweep(algo, base_hparams_grid, method_type, training_file, validation_file,
                 suffix_prefix, extra_body=None, grader=None):
    assert training_file and validation_file, f"{algo} file IDs missing"
    jobs = []
    for i, hp in enumerate(_grid_product(base_hparams_grid)):
        suffix = f"{suffix_prefix}-sw{i:02d}"
        method = {"type": method_type}
        inner = {"hyperparameters": hp}
        if grader is not None:
            inner["grader"] = grader
        method[method_type] = inner
        print(f"  [{algo} #{i}] {hp}  suffix={suffix}")
        job = openai_client.fine_tuning.jobs.create(
            model=(RFT_BASE_MODEL if method_type == "reinforcement" else SFT_DPO_BASE_MODEL),
            training_file=training_file,
            validation_file=validation_file,
            method=method,
            extra_body=(extra_body or {"trainingType": "GlobalStandard"}),
            suffix=suffix,
        )
        jobs.append({"hp": hp, "job_id": job.id, "suffix": suffix, "algo": algo})
        print(f"    -> {job.id}  status={job.status}")
    return jobs

sweep_jobs = _state.get("sweep_jobs", {})

if DO_SWEEP:
    print("=== Launching SFT sweep ===")
    sweep_jobs["sft"] = launch_sweep(
        "SFT", SWEEP_SFT_GRID, "supervised",
        sft_train_id, sft_val_id, "poker-sft-sw",
    )
    print("\n=== Launching DPO sweep ===")
    sweep_jobs["dpo"] = launch_sweep(
        "DPO", SWEEP_DPO_GRID, "dpo",
        dpo_train_id, dpo_val_id, "poker-dpo-sw",
    )
    save_state({"sweep_jobs": sweep_jobs})
    print(f"\nLaunched {len(sweep_jobs.get('sft', []))} SFT + {len(sweep_jobs.get('dpo', []))} DPO jobs.")
else:
    print(f"DO_SWEEP=False. Cached: sft={len(sweep_jobs.get('sft', []))}  dpo={len(sweep_jobs.get('dpo', []))}")

## 8. Monitor Training Progress

In [ ]:
def show_job_status(label, job_id, n_events=3):
    if not job_id:
        print(f"[{label}]  no job id")
        return None
    latest = openai_client.fine_tuning.jobs.retrieve(job_id)
    print(f"[{label}]  status: {latest.status}  id: {latest.id}")
    try:
        events = list(openai_client.fine_tuning.jobs.list_events(job_id, limit=n_events))
        for ev in events:
            print(f"    - {ev.message}")
    except Exception:
        pass
    return latest

show_job_status("SFT", sft_job_id)
show_job_status("DPO", dpo_job_id)
if RUN_RFT:
    show_job_status("RFT", rft_job_id)

if sweep_jobs.get("sft") or sweep_jobs.get("dpo"):
    print()
    for algo in ("sft", "dpo"):
        bucket = sweep_jobs.get(algo, [])
        if not bucket:
            continue
        statuses = {}
        for j in bucket:
            try:
                s = openai_client.fine_tuning.jobs.retrieve(j["job_id"]).status
                statuses[s] = statuses.get(s, 0) + 1
            except Exception:
                statuses["error"] = statuses.get("error", 0) + 1
        print(f"[{algo.upper()} sweep]  {dict(statuses)}")

### 8.1 Retrieve fine-tuned model IDs

In [ ]:
def get_finetuned_id(job_id):
    if not job_id:
        return None
    latest = openai_client.fine_tuning.jobs.retrieve(job_id)
    if latest.status == "succeeded":
        return latest.fine_tuned_model
    print(f"  {job_id}: status={latest.status}")
    return None

if TRAIN_SFT and sft_job_id:
    new_id = get_finetuned_id(sft_job_id)
    if new_id:
        sft_model_id = new_id
        save_state({"sft_model_id": sft_model_id})

if TRAIN_DPO and dpo_job_id:
    new_id = get_finetuned_id(dpo_job_id)
    if new_id:
        dpo_model_id = new_id
        save_state({"dpo_model_id": dpo_model_id})

if RUN_RFT and rft_job_id:
    new_id = get_finetuned_id(rft_job_id)
    if new_id:
        rft_model_id = new_id
        save_state({"rft_model_id": rft_model_id})

print(f"SFT model: {sft_model_id}")
print(f"DPO model: {dpo_model_id}")
print(f"RFT model: {rft_model_id}")

### 8.2 Best-of-sweep selection

Downloads each sweep job's result file, pulls final validation loss, picks the
winner. The winner overwrites `sft_model_id` / `dpo_model_id` in state — so
Section 9 deploys exactly one model per algorithm.

In [ ]:
from io import BytesIO

def _final_val_loss(job_id):
    job = openai_client.fine_tuning.jobs.retrieve(job_id)
    if job.status != "succeeded" or not job.result_files:
        return None
    content = openai_client.files.content(job.result_files[0]).read()
    df = pd.read_csv(BytesIO(content))
    val_cols = [c for c in df.columns if "valid" in c.lower() and "loss" in c.lower()]
    if not val_cols:
        return None
    vl = df[val_cols[0]].dropna()
    return float(vl.min()) if len(vl) else None

def pick_best_from_sweep(algo):
    bucket = sweep_jobs.get(algo, [])
    if not bucket:
        print(f"No {algo.upper()} sweep jobs.")
        return None, None
    scored = []
    for j in bucket:
        vloss = _final_val_loss(j["job_id"])
        print(f"  {j['suffix']:>24}  hp={j['hp']}  val_loss={vloss}")
        if vloss is not None:
            scored.append((vloss, j))
    if not scored:
        print(f"No {algo.upper()} jobs have val_loss yet.")
        return None, None
    scored.sort(key=lambda x: x[0])
    best_vloss, best_job = scored[0]
    model_id = get_finetuned_id(best_job["job_id"])
    print(f"BEST {algo.upper()}: {best_job['suffix']}  val_loss={best_vloss:.4f}")
    print(f"  model_id={model_id}")
    return model_id, best_job

if sweep_jobs.get("sft"):
    print("=== SFT sweep selection ===")
    best_id, _ = pick_best_from_sweep("sft")
    if best_id:
        sft_model_id = best_id
        save_state({"sft_model_id": sft_model_id, "sft_job_id": sft_job_id, "sft_prompt_version": PROMPT_VERSION})

if sweep_jobs.get("dpo"):
    print("\n=== DPO sweep selection ===")
    best_id, _ = pick_best_from_sweep("dpo")
    if best_id:
        dpo_model_id = best_id
        save_state({"dpo_model_id": dpo_model_id, "dpo_job_id": dpo_job_id, "dpo_prompt_version": PROMPT_VERSION})

print(f"\nWill deploy: SFT={sft_model_id}")
print(f"             DPO={dpo_model_id}")

## 9. Set Fine-Tuned Model Deployment Names

> **WARNING:** deployments cost money while alive and **auto-terminate after
> 4 hours** on this course Azure setup. Deploy → eval → cleanup must all
> finish inside one 4h window.

In [ ]:
# Input names for deployment from TA
sft_deployment = "<your-sft-deployment-name>"
dpo_deployment = "<your-dpo-deployment-name>"
rft_deployment = ""

if DEPLOY_SFT:
    save_state({"sft_deployment": sft_deployment})
if DEPLOY_DPO:
    save_state({"dpo_deployment": dpo_deployment})
if DEPLOY_RFT:
    save_state({"rft_deployment": rft_deployment})

print(f"\nActive deployments:")
print(f"  SFT: {sft_deployment}")
print(f"  DPO: {dpo_deployment}")
print(f"  RFT: {rft_deployment}")

## 10. Evaluate on PokerBench Test Set

Uses `build_messages(..., prompt_version=<per-model>)` so each model gets the
prompt regime it was trained under.

In [ ]:
# Optional: poll the deployments until they're ready before running evals.

import time

def warm_up(deployment_name, label, max_wait=600, poll=15):
    """Poll a deployment with a trivial chat call until it answers or we time out."""
    if not deployment_name:
        print(f"[{label}] no deployment name")
        return False
    start = time.time()
    while time.time() - start < max_wait:
        try:
            openai_client.chat.completions.create(
                model=deployment_name,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=1, temperature=0.0,
            )
            elapsed = int(time.time() - start)
            print(f"[{label}] ready after {elapsed}s")
            return True
        except Exception as e:
            if "DeploymentNotReady" in str(e) or "provisioningState" in str(e):
                print(f"[{label}] not ready yet, waiting {poll}s...")
                time.sleep(poll)
                continue
            print(f"[{label}] unexpected error: {e}")
            return False
    print(f"[{label}] timed out after {max_wait}s")
    return False

for label, dep in [("SFT", sft_deployment), ("DPO", dpo_deployment)]:
    warm_up(dep, label)

In [ ]:
import time, os
from datetime import datetime, timezone

PREDICTIONS_DIR = os.path.join(CACHE_DIR, f"predictions_{RUN_VERSION}")
os.makedirs(PREDICTIONS_DIR, exist_ok=True)


# =============================================================================
# Single-pass inference collector
# =============================================================================

def collect_predictions(model_deployment, model_label, rows, prompt_version,
                        flatten=False, n=None, max_retries=4, retry_wait=15,
                        warmup=True, force=False):
    """Run inference once per row; persist raw predictions + status to disk.

    Returns dict: {model_label, deployment, prompt_version, flatten, n_rows,
                   timestamp, predictions: [{row_idx, pred, status}, ...]}

    status in {"ok", "filter", "not_ready", "error"}. Only "ok" rows count in metrics.

    If a cache file exists for this label AND the run parameters match, skips
    inference and returns the cache. Pass force=True to re-eval."""
    out_path = os.path.join(PREDICTIONS_DIR, f"{model_label}.json")

    if os.path.exists(out_path) and not force:
        with open(out_path) as f:
            cached = json.load(f)
        params_match = (
            cached.get("deployment") == model_deployment
            and cached.get("prompt_version") == prompt_version
            and cached.get("n_rows") == (n or len(rows))
        )
        if params_match:
            n_ok = sum(1 for p in cached["predictions"] if p["status"] == "ok")
            print(f"[{model_label}] using cached predictions "
                  f"({len(cached['predictions'])} rows, {n_ok} ok)")
            return cached
        print(f"[{model_label}] cache params differ from current run, re-running")

    if warmup and not warm_up(model_deployment, model_label):
        print(f"[{model_label}] deployment never became ready, aborting")
        return None

    n = n or len(rows)
    predictions = []
    consecutive_not_ready = 0

    for i, r in enumerate(rows[:n]):
        rec = {
            "row_idx":          i,
            "pred":             "",
            "status":           "error",
            "correct_decision": r["correct_decision"],
            "available_moves":  r["available_moves"],
        }
        for attempt in range(max_retries):
            try:
                messages = build_messages(r["prompt"],
                                          prompt_version=prompt_version,
                                          flatten=flatten)
                resp = openai_client.chat.completions.create(
                    model=model_deployment, messages=messages,
                    max_tokens=20, temperature=0.0,
                )
                rec["pred"] = resp.choices[0].message.content
                rec["status"] = "ok"
                consecutive_not_ready = 0
                break
            except Exception as e:
                es = str(e)
                if "content_filter" in es or "ResponsibleAIPolicy" in es:
                    rec["status"] = "filter"
                    break
                if "DeploymentNotReady" in es or "provisioningState" in es:
                    if attempt < max_retries - 1:
                        time.sleep(retry_wait * (attempt + 1))
                        continue
                    rec["status"] = "not_ready"
                    consecutive_not_ready += 1
                    break
                rec["status"] = "error"
                rec["error"] = es[:200]
                break

        predictions.append(rec)

        if consecutive_not_ready >= 10:
            print(f"  Aborting at row {i}: 10 consecutive DeploymentNotReady. "
                  f"Deployment appears dead — request redeploy.")
            break

        if (i + 1) % 100 == 0:
            n_ok = sum(1 for p in predictions if p["status"] == "ok")
            print(f"  [{model_label}] progress {i+1}/{n} ({n_ok} ok so far)")

    result = {
        "model_label":    model_label,
        "deployment":     model_deployment,
        "prompt_version": prompt_version,
        "flatten":        flatten,
        "n_rows":         n,
        "timestamp":      datetime.now(timezone.utc).isoformat(),
        "predictions":    predictions,
    }
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    status_counts = {}
    for p in predictions:
        status_counts[p["status"]] = status_counts.get(p["status"], 0) + 1
    print(f"[{model_label}] saved {len(predictions)} predictions -> {out_path}")
    print(f"   status breakdown: {status_counts}")
    return result


# =============================================================================
# Pure metric helpers — all operate on (predictions_data, test_rows).
# Zero API calls. Add new helpers here without touching the inference path.
# =============================================================================

def _iter_ok(preds_data, test_rows):
    """Yield (pred_record, test_row, m_tuple, c_tuple) for status=='ok' rows."""
    for p in preds_data["predictions"]:
        if p["status"] != "ok":
            continue
        idx = p["row_idx"]
        if test_rows is not None and 0 <= idx < len(test_rows):
            r = test_rows[idx]
        else:
            r = {"correct_decision": p["correct_decision"],
                 "available_moves":  p["available_moves"],
                 "stage":            p.get("stage", "unknown"),
                 "street":           p.get("street", "unknown")}
        yield p, r, action_tuple(p["pred"]), action_tuple(r["correct_decision"])


def accuracy_from_preds(preds_data, test_rows):
    correct = scored = 0
    for p, r, _, _ in _iter_ok(preds_data, test_rows):
        scored += 1
        if actions_match(p["pred"], r["correct_decision"]):  # ← was: m == c
            correct += 1
    return (correct / max(scored, 1), correct, scored)


def stage_accuracy_from_preds(preds_data, test_rows):
    """Accuracy split by preflop vs postflop."""
    by = {"preflop": [0, 0], "postflop": [0, 0]}
    for _, r, m, c in _iter_ok(preds_data, test_rows):
        stage = r["stage"]
        by[stage][1] += 1
        if m == c:
            by[stage][0] += 1
    return {s: (co/max(t, 1), co, t) for s, (co, t) in by.items()}


def street_accuracy_from_preds(preds_data, test_rows):
    """Accuracy split by preflop / flop / turn / river."""
    streets = ["preflop", "flop", "turn", "river"]
    by = {s: [0, 0] for s in streets}
    for _, r, m, c in _iter_ok(preds_data, test_rows):
        s = r["street"]
        if s in by:
            by[s][1] += 1
            if m == c:
                by[s][0] += 1
    return {s: (co/max(t, 1), co, t) for s, (co, t) in by.items()}


def classify_from_preds(preds_data, test_rows):
    """Class A/B/C counts, plus per-status bookkeeping."""
    counts = {"class_a": 0, "class_b": 0, "class_c": 0,
              "filter": 0, "not_ready": 0, "error": 0, "scored": 0}
    for p in preds_data["predictions"]:
        st = p["status"]
        if st != "ok":
            counts[st] = counts.get(st, 0) + 1
            continue
        counts["scored"] += 1
        r = test_rows[p["row_idx"]]
        m_cat, m_amt = action_tuple(p["pred"])
        c_cat, c_amt = action_tuple(r["correct_decision"])
        if actions_match(p["pred"], r["correct_decision"]):  # ← was: m_cat==c_cat and m_amt==c_amt
            counts["class_a"] += 1
        elif m_cat == c_cat:
            counts["class_b"] += 1
        else:
            counts["class_c"] += 1
    return counts


def confusion_from_preds(preds_data, test_rows, cats=None):
    """5x5 action-category confusion matrix (rows=true, cols=pred)."""
    import numpy as np
    cats = cats or ["fold", "call", "check", "bet", "raise"]
    M = np.zeros((len(cats), len(cats)), dtype=int)
    for _, _, (m_cat, _), (c_cat, _) in _iter_ok(preds_data, test_rows):
        if c_cat in cats and m_cat in cats:
            M[cats.index(c_cat)][cats.index(m_cat)] += 1
    return M, cats


def action_distribution_from_preds(preds_data, test_rows, cats=None):
    """Action-category distribution of model outputs vs ground truth.
    Useful for spotting mode collapse (e.g., model always says 'call')."""
    cats = cats or ["fold", "call", "check", "bet", "raise"]
    m_counts = {c: 0 for c in cats}
    t_counts = {c: 0 for c in cats}
    total = 0
    for _, _, (m_cat, _), (c_cat, _) in _iter_ok(preds_data, test_rows):
        total += 1
        if m_cat in m_counts: m_counts[m_cat] += 1
        if c_cat in t_counts: t_counts[c_cat] += 1
    total = max(total, 1)
    return ({c: m_counts[c]/total for c in cats},
            {c: t_counts[c]/total for c in cats})


def bet_sizing_error_from_preds(preds_data, test_rows):
    """For Class B predictions (correct category, wrong size), mean relative
    sizing error |m_amt - c_amt| / c_amt. Characterises 'directionally right
    but off on sizing' failure modes."""
    errors = []
    for _, _, (m_cat, m_amt), (c_cat, c_amt) in _iter_ok(preds_data, test_rows):
        if (m_cat == c_cat and m_amt is not None and c_amt is not None
                and m_amt != c_amt):
            errors.append(abs(m_amt - c_amt) / max(c_amt, 1.0))
    if not errors:
        return (None, 0)
    return (sum(errors) / len(errors), len(errors))


print("Single-pass eval infrastructure ready.")
print(f"  Predictions cache dir: {PREDICTIONS_DIR}")

In [ ]:
# Set True to force re-eval even if a cache file exists (e.g., after redeploying).
FORCE_RE_EVAL = False

models_to_eval = []
if DEPLOY_SFT and sft_deployment:
    models_to_eval.append(("SFT", sft_deployment, sft_prompt_version, False))
if DEPLOY_DPO and dpo_deployment:
    models_to_eval.append(("DPO", dpo_deployment, dpo_prompt_version, False))
if DEPLOY_RFT and rft_deployment:
    models_to_eval.append(("RFT", rft_deployment, rft_prompt_version, True))

predictions_store = {}   # {model_label: full predictions dict}

if RUN_FULL_EVAL:
    for label, dep, pv, flat in models_to_eval:
        print(f"\n=== {label}: {dep}  (prompt={pv}) ===")
        res = collect_predictions(
            dep, label, test_rows,
            prompt_version=pv, flatten=flat,
            n=N_TEST, force=FORCE_RE_EVAL,
        )
        if res:
            predictions_store[label] = res
else:
    # Load from disk without hitting Azure.
    for label, dep, pv, flat in models_to_eval:
        path = os.path.join(PREDICTIONS_DIR, f"{label}.json")
        if os.path.exists(path):
            with open(path) as f:
                predictions_store[label] = json.load(f)
            n_ok = sum(1 for p in predictions_store[label]["predictions"] if p["status"] == "ok")
            print(f"[{label}] loaded {n_ok} ok predictions from cache")

# Derive the headline dicts every other cell expects. All pure functions over
# the cached predictions — never re-runs inference.
results = {}
stage_results = {}
if baseline_acc is not None:
    results["Base (no FT)"] = baseline_acc
for label, preds in predictions_store.items():
    acc, _, _ = accuracy_from_preds(preds, test_rows)
    results[label] = acc
    stage_results[label] = {s: v[0] for s, v in stage_accuracy_from_preds(preds, test_rows).items()}

print("\n=== Binary accuracy on PokerBench test ===")
for name, acc in results.items():
    print(f"  {name:>14}: {acc:.3f}")

save_state({"results": results, "stage_results": stage_results})

## 11. Analysis and Visualizations

### 11.1 Bar chart — binary accuracy

In [ ]:
import matplotlib.pyplot as plt

if results:
    names = list(results.keys())
    accs  = [results[n] for n in names]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(names, accs, color=["gray", "C0", "C1", "C2"][:len(names)])
    ax.set_ylabel("Binary Accuracy")
    ax.set_title("PokerBench GTO Action Prediction")
    ax.set_ylim(0, 1)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{acc:.3f}", ha="center")
    plt.tight_layout()
    plt.savefig(f"results_comparison_{RUN_VERSION}.png", dpi=120)
    plt.show()
    print("Saved results_comparison.png")
else:
    print("No results — run Section 10 first.")

### 11.2 Stage breakdown (preflop vs postflop)

In [ ]:
# Stage breakdown (preflop vs postflop) and finer per-street breakdown.
# Both derived from cached predictions — no API calls.

print("=== Preflop vs Postflop ===")
for label, preds in predictions_store.items():
    st = stage_accuracy_from_preds(preds, test_rows)
    pre_acc, pre_c, pre_t = st["preflop"]
    post_acc, post_c, post_t = st["postflop"]
    print(f"  {label:>4}: preflop={pre_acc:.3f} ({pre_c}/{pre_t})  "
          f"postflop={post_acc:.3f} ({post_c}/{post_t})")

print("\n=== Per-street (preflop / flop / turn / river) ===")
street_results = {}
for label, preds in predictions_store.items():
    sr = street_accuracy_from_preds(preds, test_rows)
    street_results[label] = {s: v[0] for s, v in sr.items()}
    pieces = [f"{s}={v[0]:.3f} ({v[1]}/{v[2]})" for s, v in sr.items() if v[2] > 0]
    print(f"  {label:>4}: " + "  ".join(pieces))

if street_results:
    save_state({"street_results": street_results})

### 11.3 Detailed metrics (Class A / B / C / illegal)

Mirrors the RFT training reward so the writeup can discuss partial-credit
quality alongside binary accuracy.

In [ ]:
# Class A/B/C/illegal + partial-credit score + per-status breakdown.
# Plus two new metrics from cached predictions: action distribution (for mode
# collapse detection) and mean Class-B sizing error.

rows_out = []
for label, preds in predictions_store.items():
    c = classify_from_preds(preds, test_rows)
    s = max(c["scored"], 1)
    partial = (c["class_a"] * 1.0 + c["class_b"] * 0.5) / s
    rows_out.append({
        "Model":     label,
        "Accuracy":  c["class_a"] / s,
        "Class B":   c["class_b"] / s,
        "Class C":   c["class_c"] / s,
        "Partial":   partial,
        "Filter":    c["filter"],
        "NotReady":  c["not_ready"],
        "Other err": c["error"],
        "N scored":  c["scored"],
    })

if rows_out:
    metrics_df = pd.DataFrame(rows_out).set_index("Model")
    if baseline_acc is not None:
        metrics_df.loc["Base (no FT)"] = {
            "Accuracy": baseline_acc,
            "Class B": float("nan"), "Class C": float("nan"),
            "Partial": float("nan"),
            "Filter": 0, "NotReady": 0, "Other err": 0, "N scored": N_BASELINE,
        }
        metrics_df = metrics_df.reindex(
            ["Base (no FT)"] + [m for m in metrics_df.index if m != "Base (no FT)"]
        )
    pd.options.display.float_format = "{:.3f}".format
    print("=== Detailed Metrics ===")
    print(metrics_df)
    metrics_df.to_csv(f"detailed_metrics_{RUN_VERSION}.csv")
    print("\nSaved detailed_metrics.csv")

# Action-category distribution — is any model mode-collapsing to one action?
print("\n=== Action distribution (ok predictions only) ===")
first_label = next(iter(predictions_store), None)
if first_label:
    _, truth_dist = action_distribution_from_preds(predictions_store[first_label], test_rows)
    print(f"  {'Truth':>6}: " + "  ".join(f"{c}={truth_dist[c]:.2f}" for c in truth_dist))
    for label, preds in predictions_store.items():
        model_dist, _ = action_distribution_from_preds(preds, test_rows)
        print(f"  {label:>6}: " + "  ".join(f"{c}={model_dist[c]:.2f}" for c in model_dist))

# Mean relative sizing error on Class B — how far off is each model when it's
# directionally right but wrong on size?
print("\n=== Class-B sizing error (mean |Δsize|/correct_size) ===")
for label, preds in predictions_store.items():
    mean_err, n = bet_sizing_error_from_preds(preds, test_rows)
    if n:
        print(f"  {label:>4}: mean rel error = {mean_err:.3f}  (n={n} Class B rows)")
    else:
        print(f"  {label:>4}: no Class B rows to measure")

### 11.4 Action-category confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Action-category confusion, row-normalized. Pure fn over cached predictions.
if predictions_store:
    labels = list(predictions_store.keys())
    fig, axes = plt.subplots(1, len(labels), figsize=(5*len(labels), 4))
    if len(labels) == 1:
        axes = [axes]
    for ax, label in zip(axes, labels):
        M, cats = confusion_from_preds(predictions_store[label], test_rows)
        M_norm = M / np.maximum(M.sum(axis=1, keepdims=True), 1)
        ax.imshow(M_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(cats)), cats, rotation=45)
        ax.set_yticks(range(len(cats)), cats)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{label} (row-normalized)")
        for i in range(len(cats)):
            for j in range(len(cats)):
                if M_norm[i, j] > 0.01:
                    ax.text(j, i, f"{M_norm[i, j]:.2f}", ha="center", va="center",
                            color="white" if M_norm[i, j] > 0.5 else "black",
                            fontsize=8)
    plt.tight_layout()
    plt.savefig(f"confusion_matrices_{RUN_VERSION}.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved confusion_matrices.png")
else:
    print("No predictions in store — run Section 10 first.")

### 11.5 Training loss curves

In [ ]:
from io import BytesIO

def get_training_curve(job_id):
    if not job_id:
        return None
    latest = openai_client.fine_tuning.jobs.retrieve(job_id)
    if latest.status != "succeeded" or not latest.result_files:
        print(f"  Job {job_id}: status={latest.status}, no result file")
        return None
    content = openai_client.files.content(latest.result_files[0]).read()
    return pd.read_csv(BytesIO(content))

fig, ax = plt.subplots(figsize=(7, 4))
for name, jid in [("SFT", sft_job_id), ("DPO", dpo_job_id)]:
    df = get_training_curve(jid)
    if df is None:
        continue
    loss_col = next((c for c in df.columns if "train_loss" in c.lower() or "loss" in c.lower()), None)
    step_col = "step" if "step" in df.columns else df.columns[0]
    if loss_col:
        ax.plot(df[step_col], df[loss_col], label=f"{name} train loss", alpha=0.8)

ax.set_xlabel("Training step"); ax.set_ylabel("Loss")
ax.set_title("Training loss curves")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves_{RUN_VERSION}.png", dpi=120)
plt.show()
print("Saved training_curves.png")

## 12. Cleanup — Remove Deployments

**Always delete deployments after evaluation.** Let TA's know